In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import anndata as ad
import numpy as np

import torch

from sckitflow import predict_adata
from sckitflow.core._types import PredictionData
from sckitflow.core.methods import ODEInference, ODEInferenceConfig
from sckitflow.core.nn import MLPVelocity
from sckitflow.core.surrogate import SurrogatePotential
from sckitflow.data import DataManagerConfig, FlowDataModule

/Users/lorenzo.consoli/micromamba/envs/scflow13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
adata_path = "/Users/lorenzo.consoli/Work/data/collab-goettgens/loop0/data/adata_500k.h5ad"
adata = ad.read_h5ad(adata_path)
adata = adata[np.random.default_rng(0).choice(adata.n_obs, 500, replace=False)].copy()
adata

AnnData object with n_obs × n_vars = 500 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'leiden', 'leiden_sub', 'leiden_old', 'cell_type'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'leiden_colors_old', 'm

In [4]:
molecules = [
    "740-yp_[µm]",
    "mtg_[µm]",
    "rhflt3l_[ng_ml]",
    "gm-csf_[ng_ml]",
    "tpo_[ng_ml]",
    "sr1_[nm]",
    "um171_[nm]",
    "um729_[µm]",
    "scf_[ng_ml]",
    "butyzamide_[nm]",
    "retinoic_acid_[µm]",
    "ldl_[ng_ml]",
    "il3_[ng_ml]",
    "o2_[%]",
    "days_of_culture",
]
molecules_vals = np.log1p(adata.obs[molecules].astype(float).values)
condition_key = "medium_log1p"
adata.obsm[condition_key] = molecules_vals

In [5]:
scatter_features = [
    "FSC-A :: FSC - Area",
    "FSC-H :: FSC - Height",
    "FSC-W :: FSC - Width",
    "SSC-A :: SSC - Area",
    "SSC-H :: SSC - Height",
    "SSC-W :: SSC - Width",
]
X_scatter = adata.obs[scatter_features].values
X_joint = np.concatenate([adata.X, X_scatter], axis=1)

X_mean = X_joint.mean(0)
X_std = X_joint.std(0)
X = (X_joint - X_mean) / X_std

state_key = "X_std"
adata.obsm[state_key] = X

In [6]:
dm = DataManagerConfig(
    sample_rep=state_key,
    conditions_covariates=[condition_key],
    # groups=["source_id"],
    # groups_encoding={"source_id": "one-hot"},
).build()
dmod = FlowDataModule.from_adata(
    adata,
    dm,
    n_train_steps=2_000,
    batch_size=128,
)
dmod.set_predict_data(
    adata,
    max_per_group=10_000,
    require_target_state=False,
)

In [13]:
module = MLPVelocity(
    dmod.data_dims.state_dim,
    time_features_id="torch-cfm",
    conditioning_id="resnet1d",
    vf_decoder_mlp_kwargs={"hidden_dims": [32, 32]},
    condition_encoder_input_layers={
        **{
            cond: {"input_dim": cond_dim, "output_dim": 2}
            for cond, cond_dim in dmod.data_dims.condition_continuous_dims.items()
        },
        **{
            cond: {"input_dim": cond_dim, "output_dim": 2} for cond, cond_dim in dmod.data_dims.groups_reps_dims.items()
        },
    },
)
inference = ODEInference(
    module,
    ODEInferenceConfig(generate_from_noise=True, n_samples=2),
    dims=dmod.data_dims,
)

adata_pred = predict_adata(dmod, inference, adata)
adata_pred

Predicting: 100%|██████████| 1/1 [00:00<00:00,  6.99it/s]


AnnData object with n_obs × n_vars = 500 × 26
    obs: '_sckitflow_all_cells'
    obsm: 'raw_samples', 'medium_log1p'
    layers: None (.X)

In [22]:
class MeanMarkerMSE(SurrogatePotential):
    """Class for mean marker regression."""

    def get_response(self, pred_data: PredictionData) -> torch.Tensor:
        """Computes the response by averaging over the generated samples."""
        return pred_data.raw_samples.mean(0)

    def compute_raw_potential(self, y: torch.Tensor, ystar: torch.Tensor):
        """Simple MSE loss against the target expression."""
        return torch.sum((y - ystar) ** 2, dim=-1)


ystar = torch.ones((X.shape[-1],))
potential = MeanMarkerMSE(inference, ystar, dmod)

N = 10
pot, grad_fn = jvp_out = torch.func.vjp(
    potential.forward,
    {condition_key: torch.zeros((N, len(molecules)))},
)
grad = grad_fn(torch.ones_like(pot))[0]["medium_log1p"]
print(pot.shape, grad.shape)

torch.Size([10]) torch.Size([10, 15])
